<a href="https://colab.research.google.com/github/ounospanas/AIDL_B_CS01/blob/main/custom_retrieval_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Install required packages
!pip install langchain-community langchain-text-splitters pypdf sentence-transformers numpy


### 1. Load and segment a PDF

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
online_document = "https://academicweb.nd.edu/~lemmon/courses/deep-learning/lecture-book/deep-learning-book-2025.pdf"

loader = PyPDFLoader(online_document)
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print(f"First page preview:\n{pages[0].page_content[:500]}")

Loaded 383 pages
First page preview:
Introduction to Deep Learning
February 13, 2025
Department of Electrical Engineering
University of Notre Dame
1


In [7]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(pages)
print(chunks,'chunks')

print(f"\nCreated {len(chunks)} chunks")
print(f"First chunk:\n{chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-13T11:58:41-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-02-13T11:58:41-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'source': 'https://academicweb.nd.edu/~lemmon/courses/deep-learning/lecture-book/deep-learning-book-2025.pdf', 'total_pages': 383, 'page': 0, 'page_label': '1'}, page_content='Introduction to Deep Learning\nFebruary 13, 2025\nDepartment of Electrical Engineering\nUniversity of Notre Dame\n1'), Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-13T11:58:41-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-02-13T11:58:41-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 

### 2. Embed and store in numpy array

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load an open embedding model (runs locally, no API key needed)
model_emb = SentenceTransformer('all-MiniLM-L6-v2') # 384 dimensions

# Extract text from chunks
texts = [document.page_content for document in chunks]
print(texts,'texts')

# Generate embeddings
embeddings = model_emb.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings,'embeddings')
print(type(embeddings))

# Convert to numpy array (it's probably already a numpy array, but being explicit)
embeddings_array = np.array(embeddings)

# Save to disk
np.save("chunk_embeddings.npy", embeddings_array)

# Load later with:
embeddings_array = np.load("chunk_embeddings.npy")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

['Introduction to Deep Learning\nFebruary 13, 2025\nDepartment of Electrical Engineering\nUniversity of Notre Dame\n1', 'Contents\nPreface v\nChapter 1. Introduction - Learning by Example 1\n1. Problem Statement 1\n2. Types of Learning-by Example Problems 4\n2.1. Binary Classification: 4\n2.2. Regression Problem: 6\n2.3. Logistic Regression 7\n3. Neural Network Model Sets 10\n3.1. Perceptron: 11\n3.2. Multi-layer Perceptron: 13\n3.3. Deep Neural Networks: 14\n4. Deep Learning Software Libraries 16\nChapter 2. Generalization - a statistical approach 25\n1. Infeasibility of Perfect Learning 26\n2. PAC Learning 28\n3. Concentration Inequalities 31\n4. Generalization Ability of Finite Model Sets 34\n5. Growth Function for Infinite Model Sets 39\n6. Generalization Ability of Infinite Model Sets 43\n7. Bias-Variance Tradeoff and Early Stopping 47\nChapter 3. Neural Network Model Sets 53\n1. Perceptron 53\n2. Multi-layer Perceptrons and Deep Neural Networks 58\n3. Universal Approximation Abil

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

[[-0.08902965 -0.0242339   0.14901336 ...  0.03317125 -0.07673179
   0.00364158]
 [-0.00368542 -0.06015948  0.04258726 ... -0.04156267 -0.1172724
   0.03371109]
 [-0.04087811 -0.03186905  0.04943478 ... -0.01458264 -0.07716706
  -0.03837027]
 ...
 [-0.02406419 -0.02328431  0.05628271 ...  0.01139295 -0.1246227
   0.02216812]
 [-0.14470932  0.00271949  0.04614756 ... -0.02418032 -0.07426975
  -0.02219883]
 [-0.13785748 -0.09386458  0.05449985 ... -0.02580137 -0.07859588
   0.04491147]] embeddings
<class 'numpy.ndarray'>


### 3. use an S/LLM from Hugginface for paraphrase

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

question = "What is deep learning and how does it work?"

prompt = (
    f"Rewrite the following question in two different ways.\n"
    f"Do not answer it.\n\n"
    f"Question: {question}\n\n"
    f"Paraphrase 1:"
)


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

lines = [l.strip() for l in generated_text.split("\n") if l.strip()]
paraphrases = [l for l in lines if l.startswith("Paraphrase")]

paraphrases = [p.split(":", 1)[-1].strip() for p in paraphrases][:2]

print(paraphrases)


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

['Describe the concept of deep learning, and explain its underlying principles.', 'Tell me about deep learning and its ability to learn complex patterns in data.']


### 4. Retrieve 5 most semantically close chunks (cosine similarity) for every paraphrase*, then add threshold 0.3 and select top 3

In [13]:
from sklearn.metrics.pairwise import cosine_similarity


question= "What is deep learning and how does it work?"
queries = [
    question,
    paraphrases[0],
    paraphrases[1]
]

query_embeddings = model_emb.encode(
    queries,
    normalize_embeddings=True
)

similarities = cosine_similarity(query_embeddings, embeddings_array)

threshold = 0.3

for i in range(len(queries)):

    print("\nQuery:", queries[i])

    similarity_scores = similarities[i]

    #top 5
    top5 = similarity_scores.argsort()[-5:][::-1]

    #apply threshold
    filtered = [id for id in top5 if similarity_scores[id] >= threshold]

    #select top 3
    top3_indices = filtered[:3]

    for idx in top3_indices:
        print(f"Chunk {idx} → cosine similarity: {similarity_scores[idx]:.4f}")

    if len(top3_indices) > 0:
        avg_score = sum(similarity_scores[idx] for idx in top3_indices) / len(top3_indices)
        print(f"Average Top-3 Cosine Similarity: {avg_score:.4f}")




Query: What is deep learning and how does it work?
Chunk 249 → cosine similarity: 0.5910
Chunk 46 → cosine similarity: 0.5654
Chunk 10 → cosine similarity: 0.5581
Average Top-3 Cosine Similarity: 0.5715

Query: Describe the concept of deep learning, and explain its underlying principles.
Chunk 12 → cosine similarity: 0.6075
Chunk 10 → cosine similarity: 0.5887
Chunk 3 → cosine similarity: 0.5858
Average Top-3 Cosine Similarity: 0.5940

Query: Tell me about deep learning and its ability to learn complex patterns in data.
Chunk 12 → cosine similarity: 0.5695
Chunk 10 → cosine similarity: 0.5608
Chunk 44 → cosine similarity: 0.5579
Average Top-3 Cosine Similarity: 0.5627


### 5. Apply the augmentation phase using an SLM (could be the one you have used in step 2 or the only aligned with DPO)

In [17]:
retrieved_chunks = [chunks[id].page_content for id in top3_indices]

context = "\n\n".join(retrieved_chunks)

final_prompt = f"""
You are a helpful assistant.

Use ONLY the information provided in the context below to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""

inputs = tokenizer(final_prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=500,
       do_sample=True,
    temperature=0.2,
    top_p=0.9
)

final_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

final_answer = final_answer.split("Answer:")[-1].strip()

print('Final Answer: ',final_answer)



Final Answer:  Deep learning is a supervised learning problem that is often referred to as learning-by-example. It is a transformation in how we build autonomous machines.
